# PD-001 — CPU engineering only

Run all cells to obtain the parameter inventory and synthetic CPU Gate 0. No GPU training, TinyStories download, confirmation data, secrets or result publication is authorized. Gate 1 remains PROPOSED_NOT_REGISTERED.


In [ ]:
from pathlib import Path
import subprocess, sys
MODE = 'preflight'  # CPU engineering only: audit | params | dry-run | preflight
SOURCE_COMMIT = 'c7ff3a4aee6e5e2b3353b7d2b6a059e93ad0b834'
ROOT = Path('/kaggle/working/minicells-pd001-' + SOURCE_COMMIT[:12])
if MODE not in ('audit','params','dry-run','preflight'):
    raise RuntimeError('Only CPU engineering modes are authorized')
if SOURCE_COMMIT.startswith('__'):
    raise RuntimeError('Import the committed notebook with its pinned source')
ROOT.mkdir(parents=True,exist_ok=True)
def git(*args):
    print('Setup: git ' + ' '.join(args),flush=True)
    subprocess.run(['git',*args],cwd=ROOT,check=True)
ready = False
if (ROOT/'.git').exists():
    head = subprocess.run(['git','rev-parse','HEAD'],cwd=ROOT,text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE)
    ready = head.returncode == 0 and head.stdout.strip() == SOURCE_COMMIT and (ROOT/'research/projects/cellular-language-models/experiments/006-path-first-clm/extension-parameter-depth/run_pd001.py').exists()
if not ready:
    if not (ROOT/'.git').exists():
        git('init')
        git('remote','add','origin','https://github.com/ArcheLabs/mini-cells.git')
    origin = subprocess.check_output(['git','remote','get-url','origin'],cwd=ROOT,text=True).strip()
    if origin != 'https://github.com/ArcheLabs/mini-cells.git':
        raise RuntimeError('Unexpected source checkout origin')
    git('config','remote.origin.promisor','true')
    git('config','remote.origin.partialclonefilter','blob:none')
    git('sparse-checkout','init','--no-cone')
    git('sparse-checkout','set','--no-cone','/pyproject.toml','/src/','/shared/python/',
        '/research/projects/cellular-language-models/experiments/006-path-first-clm/',
        '/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/',
        '/tests/research/path_first_pd001/')
    git('fetch','--depth=1','--filter=blob:none','origin',SOURCE_COMMIT)
    git('checkout','--detach',SOURCE_COMMIT)


In [ ]:
ENTRY = ROOT/'research/projects/cellular-language-models/experiments/006-path-first-clm/extension-parameter-depth/run_pd001.py'
if not ENTRY.exists():
    raise RuntimeError('Checkout is incomplete; preserve output, remove only the incomplete code checkout and retry')
print('CPU-only PD-001 engineering preflight; no GPU or secrets',flush=True)
import os, time
out = Path('/kaggle/working/pd001-engineering')/str(time.time_ns())
env = {**os.environ, 'CUDA_VISIBLE_DEVICES': '', 'OMP_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1'}
subprocess.run([sys.executable,str(ENTRY),'--mode',MODE,'--device','cpu','--output',str(out)],cwd=ROOT,env=env,check=True)
print({'status':'CPU_ENGINEERING_COMPLETED','output':str(out),'formal_training':'NOT_REGISTERED'},flush=True)
